<a href="https://colab.research.google.com/github/thearmorr/FOOTBALL-BETTING-BOT/blob/main/FOTTBALL_BETT%C4%B0NG_BOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-generativeai
from rich.console import Console
from rich.table import Table
from rich.text import Text
import textwrap
import google.generativeai as genai
from IPython.display import display
from google.colab import userdata
import pandas as pd
import sqlite3
from sklearn.metrics import pairwise_distances
import numpy as np
from google.colab import files
from sklearn.ensemble import RandomForestClassifier
import time
import os


def to_markdown(text):
    text = text.replace("•", "  *")
    return Markdown(textwrap.indent(text, "> ", predicate=lambda _: True))
    # Or use `os.getenv('GEMINI_API_KEY')` to fetch an environment variable.

DB_FILE = 'match_data.db'

def create_database():
    """Veritabanı oluşturuluyor ve tablolar ekleniyor."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute('''CREATE TABLE IF NOT EXISTS matches (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        home_odds REAL,
        draw_odds REAL,
        away_odds REAL,
        result INTEGER
    )''')

    conn.commit()
    conn.close()

def add_data(home_odds, draw_odds, away_odds, result):
    """Yeni veriyi veritabanına ekler."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    try:
        home_odds = float(home_odds)
        draw_odds = float(draw_odds)
        away_odds = float(away_odds)
        result = int(result)
    except ValueError:
        print(f"Invalid data: home_odds={home_odds}, draw_odds={draw_odds}, away_odds={away_odds}, result={result}")
        return

    cursor.execute('''INSERT INTO matches (home_odds, draw_odds, away_odds, result)
    VALUES (?, ?, ?, ?)''', (home_odds, draw_odds, away_odds, result))

    conn.commit()
    conn.close()

def load_data():
    """Veritabanından veriyi yükler."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute('SELECT home_odds, draw_odds, away_odds, result FROM matches')
    data = cursor.fetchall()
    conn.close()
    return data

# Ajan 1: Kullanıcının Ev Sahibi Oranına Göre Maçları Filtreleme

# Agent 1 - Filtreleme ve Seçim
def agent_1(home_odds, data, tolerance=0.01):
    df = pd.DataFrame(data, columns=['home_odds', 'draw_odds', 'away_odds', 'result'])
    filtered_matches = df[abs(df['home_odds'] - home_odds) <= tolerance]



    return filtered_matches


def agent_2(user_odds, filtered_matches):
    if filtered_matches.empty:
        print("Agent 3: Geçerli filtrelenmiş maç bulunamadı.")
        return None

    # Verilerden X ve y'yi oluşturma
    X = filtered_matches[['home_odds', 'draw_odds', 'away_odds']].values
    y = filtered_matches['result']

    # Kullanıcının girdiği oranları 2D bir array haline getirme
    user_odds = np.array(user_odds).reshape(1, -1)

    # Euclidean mesafesini hesaplayarak kullanıcının oranına en yakın maçları buluyoruz
    distances = pairwise_distances(user_odds, X, metric='euclidean')[0]

    # En yakın K maçları seçiyoruz (örneğin 60)
    K = 10
    closest_indices = distances.argsort()[:K]

    # En yakın K maçın sonuçlarını alıyoruz
    closest_results = y.iloc[closest_indices]

    # Tahmin edilen sonuçları sayalım
    result_counts = {0: 0, 1: 0, 2: 0}

    for result in closest_results:
        result_counts[result] += 1

    # Her sonucun olasılığını hesaplama
    total_results = sum(result_counts.values())
    probabilities = [result_counts[i] / total_results for i in range(3)]

    return probabilities



# Agent 3 - Veri Setinin Geri Kalanıyla Yakın Oran Analizi

def agent_3(user_odds, filtered_matches):
    if filtered_matches.empty:
        print("Agent 3: Geçerli filtrelenmiş maç bulunamadı.")
        return None

    # Verilerden X ve y'yi oluşturma
    X = filtered_matches[['home_odds', 'draw_odds', 'away_odds']].values
    y = filtered_matches['result']

    # Kullanıcının girdiği oranları 2D bir array haline getirme
    user_odds = np.array(user_odds).reshape(1, -1)

    # Euclidean mesafesini hesaplayarak kullanıcının oranına en yakın maçları buluyoruz
    distances = pairwise_distances(user_odds, X, metric='euclidean')[0]

    # En yakın K maçları seçiyoruz (örneğin 60)
    K = 60
    closest_indices = distances.argsort()[:K]

    # En yakın K maçın sonuçlarını alıyoruz
    closest_results = y.iloc[closest_indices]

    # Tahmin edilen sonuçları sayalım
    result_counts = {0: 0, 1: 0, 2: 0}

    for result in closest_results:
        result_counts[result] += 1

    # Her sonucun olasılığını hesaplama
    total_results = sum(result_counts.values())
    probabilities = [result_counts[i] / total_results for i in range(3)]

    return probabilities



def agent_4(user_odds, data):
    df = pd.DataFrame(data, columns=['home_odds', 'draw_odds', 'away_odds', 'result'])
    X = df[['home_odds', 'draw_odds', 'away_odds']].values
    y = df['result']

    model = RandomForestClassifier(random_state=42, n_estimators=100)
    model.fit(X, y)

    probabilities = model.predict_proba([user_odds])[0]

    return probabilities  # Artık doğrudan olasılıkları döndürüyoruz


def agent_5(agent_2_result_prob, agent_3_result_prob, agent_4_result_prob, user_odds, data):

    combined_probabilities = [
        (agent_2_result_prob[0] + agent_3_result_prob[0] + agent_4_result_prob[0]) / 3,
        (agent_2_result_prob[1] + agent_3_result_prob[1] + agent_4_result_prob[1]) / 3,
        (agent_2_result_prob[2] + agent_3_result_prob[2] + agent_4_result_prob[2]) / 3,
    ]


    # Normalizasyon
    normalized_probabilities = [prob / sum(combined_probabilities) for prob in combined_probabilities]


    # Yüzdelik oranları yazdırma
    results_map = {0: "Beraberlik", 1: "Ev Sahibi Kazanır", 2: "Deplasman Kazanır"}
    console = Console()
    table = Table(title="HESAPLANIYOR...", style="bold red")
    table.add_column("Sonuç", style="bold white")
    table.add_column("Olasılık", justify="right", style="bold yellow")

    for i, prob in enumerate(normalized_probabilities):
        table.add_row(results_map[i], f"{prob * 100:.2f}%")

    console.print(table)

    # Nihai karar
    max_prob = max(normalized_probabilities)
    second_max_prob = sorted(normalized_probabilities, reverse=True)[1]
    threshold = 0.3  # Threshold ile yakın olasılıkları değerlendir

    if max_prob - second_max_prob < threshold:
        likely_results = [i for i, prob in enumerate(normalized_probabilities) if prob >= max_prob - threshold]
        final_result = likely_results[0] if likely_results else None
    else:
        final_result = normalized_probabilities.index(max(normalized_probabilities))


    # Sonucu yazdır
    ai_analysis(final_result, normalized_probabilities)

    return final_result, normalized_probabilities



# Global değişkenler
short_term_memory = []  # Son mesajları içerir (aktif bağlam)

def add_to_short_term_memory(user_message, bot_response):
    """Mesajları kısa süreli hafızaya ekle."""
    short_term_memory.append({"user": user_message, "bot": bot_response})
    if len(short_term_memory) > 10:  # Hafızayı 10 mesajla sınırlandır
        short_term_memory.pop(0)

def get_short_term_context():
    """Kısa süreli hafızayı bağlam olarak al."""
    return "\n".join([f"Kullanıcı: {msg['user']}\nBot: {msg['bot']}" for msg in short_term_memory])

def ai_analysis(final_result, normalized_probabilities):
    """Tahmin sonucunu açıklamak için AI tabanlı analiz API'yi kullan."""
    result_map = {0: "Beraberlik", 1: "Ev Sahibi Kazanır", 2: "Deplasman Kazanır"}
    result = result_map[final_result]

    # API'ye gönderilecek veriyi hazırlıyoruz
    data_to_send = {
        "result": result,
        "home_win_prob": normalized_probabilities[1] * 100,
        "draw_prob": normalized_probabilities[0] * 100,
        "away_win_prob": normalized_probabilities[2] * 100
    }

    # API yapılandırmasını yapıyoruz
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    genai.configure(api_key=GOOGLE_API_KEY)



    try:
        # Modeli seçiyoruz ve analiz için istek gönderiyoruz
        model = genai.GenerativeModel("gemini-1.5-flash")  # Burada doğru modelin ismini kullanmalısınız

        # Modeli, verilen talimatla yönlendiriyoruz
        analysis_prompt = f"""
        Modelin vermiş olduğu analizlerin sonucunu yüzdeliklerine bakarak kimin kazanacağını kısa ve öz ama biraz detaylı bir şekilde değerlendir.
        Sonuçlar:
        Ev Sahibi Kazanma Olasılığı: {normalized_probabilities[1] * 100:.2f}%
        Beraberlik Olasılığı: {normalized_probabilities[0] * 100:.2f}%
        Deplasman Kazanma Olasılığı: {normalized_probabilities[2] * 100:.2f}%
        """

        # Hafıza ile birlikte bağlam hazırlıyoruz
        short_term_context = get_short_term_context()
        full_prompt = f"{short_term_context}\n\n{analysis_prompt}"

        # API'ye uygun formatta içerik gönderiyoruz
        response = model.generate_content({
            "parts": [{"text": full_prompt}]
        })

        # API'den gelen yanıtı yazdırmak
        analysis_output = response.text
        print(f"\nBot 🤖: {analysis_output}")  # Tek bir çıktı verilecek

        # Hafızaya ekliyoruz
        add_to_short_term_memory(analysis_prompt, analysis_output)

        # AI analiz sonucundan sonra sohbet başlatılır
        continue_chat_with_bot()

    except Exception as e:
        print(f"API isteği sırasında bir hata oluştu: {e}")

def generate_custom_prompt(user_input):
    """
    Kullanıcının girdisine dayalı özel bir analiz açıklaması oluşturur.
    """
    custom_description = f"""
    Kullanıcının şuan vermiş olduğu veriyi geçmiş verilerdeki bilgilere dayanarak, bütün olasılıklarını derinlemesine analiz et.
    Analizinde yüzdeleri, olasılıkları değerlendir.
    Analizinden çıkardığın en olasılığı yüksek tahminleri net ve kısa bir şekilde ifade et.
    Tahmini anlaşılır, samimi ve sıcak bir dilde yap.Sen bir İddia botu gibi davran.
    ve 'Ancak, futbolda sürprizler olabilir','Ancak, futbolda kesinlik yoktur. Bu sadece bir tahmindir, sonuç tamamen farklı olabilir'gibi ifadeleri söyleme direkt tahminlerini söyle.

    Kullanıcı sorusu: {user_input}
    """
    return custom_description

is_chat_ended = False

def continue_chat_with_bot():
    """Kullanıcı ile AI modeli arasında sohbet başlat."""
    global is_chat_ended  # global değişkeni kullanıyoruz

    if is_chat_ended:  # Eğer sohbet bitmişse, herhangi bir işlem yapılmasın
        return

    print("\nBot 🤖: Tahminim hakkında konuşmak istersen buradayım! Bana istediğin soruyu sorabilirsin.")
    print("-" * 300)
    while True:
        user_input = input("Sen 🗣️: ")
        print("-" * 300)

        # 'çık' komutuyla sohbeti sonlandır
        if user_input.lower() in ['çık', 'q', 'quit', 'exit']:
            if not is_chat_ended:  # Sohbet bitmemişse
                print("Bot 🤖: Sohbeti sonlandırıyorum. Görüşürüz!")
                is_chat_ended = False  # Sohbeti sonlandır
            break
        else:
            try:


                # Kullanıcı sorusunu AI modeline iletmek
                short_term_context = get_short_term_context()
                custom_prompt = generate_custom_prompt(user_input)
                prompt = f"{short_term_context}\n\n{custom_prompt}"

                # Buradaki bağlam, her yeni mesajla birlikte dinamik olarak güncelleniyor
                response = genai.GenerativeModel("gemini-1.5-flash").generate_content({
                    "parts": [{"text": prompt}]
                })
                bot_response = response.text
                print(f"\nBot 🤖: {bot_response}")
                print("-" * 300)

                # Hafızaya ekle
                add_to_short_term_memory(user_input, bot_response)

            except Exception as e:
                print(f"Bot 🤖: Bir hata oluştu: {e}")



if __name__ == "__main__":
    create_database()

    print("Veritabanı ve tablo oluşturuldu. Veri girilmeye hazır.")

    print("Verilerinizi yükleyin (Excel veya CSV):")
    uploaded = files.upload()

    filename = list(uploaded.keys())[0]
    if filename.endswith('.csv'):
        data = pd.read_csv(filename)
    elif filename.endswith('.xlsx'):
        data = pd.read_excel(filename, engine='openpyxl')
    else:
        raise ValueError("Sadece CSV veya Excel formatları destekleniyor.")

    # Sütun isimlerini temizle
    data.columns = data.columns.str.strip()

    # Veritabanına aktarma
    for i, row in data.iterrows():
        add_data(row['home_odds_col'], row['draw_odds_col'], row['away_odds_col'], row['result_col'])

    # Sonsuz döngüde tahmin al, 'Q' tuşuna basılınca dur
    while True:
        try:
            # Veritabanını yükle
            data = load_data()

            home_odds = float(input("Ev Sahibi Oranı: "))
            draw_odds = float(input("Beraberlik Oranı: "))
            away_odds = float(input("Deplasman Oranı: "))

            # Ajanları çalıştır
            filtered_matches = agent_1(home_odds, data)

            # Agent 2 ve Agent 3'ü çalıştır
            agent_2_result = agent_2([home_odds, draw_odds, away_odds], filtered_matches)
            agent_3_result = agent_3([home_odds, draw_odds, away_odds], filtered_matches)

            # Agent 4'ün kararını al
            agent_4_result = agent_4([home_odds, draw_odds, away_odds], data)

            # Agent 5 ile nihai tahmini yap
            final_result,normalized_probabilities = agent_5(agent_2_result, agent_3_result, agent_4_result, [home_odds, draw_odds, away_odds], data)
            ai_analysis(final_result, normalized_probabilities)
            # Eğer final_result None değilse, sonucu yazdır
            if final_result is not None:
                result_mapping = {0: "Beraberlik", 1: "Ev Sahibi Kazanır", 2: "Deplasman Kazanır"}

            else:
                print("Tahmin belirlenemedi.")

        except ValueError:
            print("Geçersiz veri girdiniz. Tekrar deneyin.")

        # Çıkmak için 'Q' tuşuna bas, devam etmek için başka bir tuşa bas
        exit_option = input("\nÇıkmak için 'Q' tuşuna bas, devam etmek için başka bir tuşa bas: ")
        if exit_option.lower() == 'q':
            print("Çıkılıyor...")
            break